# Loan Default Prediction using a Decision Tree Classifier
**Machine Learning – Class Assignment**

This notebook follows all 12 steps in the assignment. Run the cells from top to bottom. Written answers are included in Markdown cells; numerical results and conclusions that depend on the dataset are generated from the actual outputs.

## Step 1: Import Libraries and Load the Data

We import the required libraries and load `loan_default.csv` into a DataFrame called `df`. We inspect the shape, data types, and first ten rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

df = pd.read_csv("loan_default.csv")

print("Dataset shape:", df.shape)
print("\\nColumn information:")
df.info()
print("\\nFirst 10 rows:")
display(df.head(10))

**Write down – Answer:** The original dataset contains **1,012 rows and 18 columns**. Numeric columns are `age`, `income`, `loan_amount`, `credit_score`, `employment_years`, `debt_to_income`, `existing_loans`, `dependents`, `bank_balance`, `interest_rate`, and `previous_defaults`. Text/categorical columns are `customer_id`, `employment_type`, `marital_status`, `education`, `loan_purpose`, `city_tier`, and `loan_default`. `customer_id` is an identifier and `loan_default` is the target.

## Step 2: Understand the Data

We inspect numeric summary statistics, categorical value counts, and the distribution of the target class.

In [ ]:
print("Numeric summary statistics:")
display(df.describe())

categorical_cols = df.select_dtypes(include=["object", "category"]).columns
for col in categorical_cols:
    print(f"\\nValue counts for {col}:")
    display(df[col].value_counts(dropna=False))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="loan_default", order=df["loan_default"].value_counts().index)
plt.title("Distribution of Loan Default")
plt.xlabel("Loan Default")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

print("Target proportions:")
display(df["loan_default"].value_counts(normalize=True).rename("proportion"))

**Write down – Answer:** There are **825 No (81.5%)** and **187 Yes (18.5%)** records in the original CSV, so the target is imbalanced. A model that mostly predicts `No` could still achieve high accuracy while missing defaulters. For that reason, we also evaluate default-class precision, recall, F1-score, ROC-AUC, and the confusion matrix.

## Step 3: Clean the Data

We count missing values, fill missing categorical values with the most frequent category, remove exact duplicate rows, and treat impossible negative age/income values as missing before imputing them with medians. We do not remove records only because `customer_id` repeats; the assignment specifically asks to remove duplicate rows.

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum().sort_values(ascending=False))

print("Exact duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Treat impossible values as missing
for col in ["age", "income", "loan_amount", "credit_score", "employment_years",
            "debt_to_income", "existing_loans", "dependents", "bank_balance",
            "interest_rate", "previous_defaults"]:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan

# Fill categorical missing values with each column's mode
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    if df[col].isna().any():
        mode_values = df[col].mode(dropna=True)
        fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
        df[col] = df[col].fillna(fill_value)

# Fill numeric missing values with the median
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("Missing values after cleaning:")
display(df.isnull().sum().sort_values(ascending=False))
print("Shape after removing exact duplicates:", df.shape)
print("Remaining negative age values:", (df["age"] < 0).sum())
print("Remaining negative income values:", (df["income"] < 0).sum())

**Write down – Answer:** The original CSV has **61 missing values in `education`**, **11 exact duplicate rows**, **5 negative ages**, and **5 negative incomes**. The cleaning code removes exact duplicate rows, fills missing categorical values with the mode, and treats impossible negative numeric values as missing before replacing them with the numeric median. After exact duplicate removal, the dataset has **1,001 rows**. These choices retain usable records while avoiding impossible values.

## Step 4: Explore the Data Visually

We compare credit scores by target class, calculate default rates by city tier and employment type, and plot correlations among numeric variables.

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="loan_default", y="credit_score", order=["No", "Yes"])
plt.title("Credit Score by Loan Default Status")
plt.tight_layout()
plt.show()

city_rates = df.groupby("city_tier")["loan_default"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
employment_rates = df.groupby("employment_type")["loan_default"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
city_rates.plot(kind="bar", ax=axes[0])
axes[0].set_title("Default Rate by City Tier")
axes[0].set_ylabel("Default rate")
axes[0].tick_params(axis="x", rotation=30)
employment_rates.plot(kind="bar", ax=axes[1])
axes[1].set_title("Default Rate by Employment Type")
axes[1].set_ylabel("Default rate")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

numeric_for_corr = df.select_dtypes(include=np.number)
plt.figure(figsize=(11, 8))
sns.heatmap(numeric_for_corr.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numeric Columns")
plt.tight_layout()
plt.show()

print("Mean numeric values by loan_default:")
display(df.groupby("loan_default")[numeric_for_corr.columns].mean().T)

**Write down – Answer:** In the cleaned data, the average credit score is about **601.5 for defaulters** and **662.9 for non-defaulters**, so credit score visibly separates the groups. The average loan amount is also higher among defaulters (about **283,606**) than non-defaulters (about **237,339**). These are useful candidate features to inspect; the plots and feature importance will show whether the patterns remain useful for prediction. Association does not establish causation.

## Step 5: Prepare the Data for Modelling

We remove the customer identifier, encode the target as `Yes = 1` and `No = 0`, convert categorical features to dummy variables, and separate features (`X`) from target (`y`).

In [ ]:
df_model = df.drop(columns=["customer_id"]).copy()
df_model["loan_default"] = df_model["loan_default"].map({"Yes": 1, "No": 0})

if df_model["loan_default"].isna().any():
    raise ValueError("Unexpected target labels found. Check loan_default values.")

df_model = pd.get_dummies(df_model, drop_first=True, dtype=int)
X = df_model.drop(columns=["loan_default"])
y = df_model["loan_default"].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target labels:", sorted(y.unique()))
display(X.head())

**Write down – Answer:** A scikit-learn Decision Tree cannot use text categories directly because its estimator expects numeric feature values. `pd.get_dummies()` converts categories into numeric indicator columns. `customer_id` is excluded because it is an identifier rather than a meaningful predictive feature. The target is encoded as 1 for default and 0 for no default.

## Step 6: Split into Training and Test Sets

We use an 80/20 split, stratifying by the target so that the default/non-default proportions remain similar in both sets. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("\\nTraining target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("Test target proportions:")
display(y_test.value_counts(normalize=True).rename("proportion"))

**Write down – Answer:** The training set is used to learn patterns, while the test set is held back for an independent evaluation on data the model has not seen during training. Training on the test set would leak information and make the evaluation overly optimistic. Stratification preserves the class proportions as closely as possible.

## Step 7: Train a Decision Tree

We train the default `DecisionTreeClassifier` with `random_state=42`, then compare training and test accuracy.

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_train_pred_dt = dt.predict(X_train)
y_test_pred_dt = dt.predict(X_test)

train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt)

print(f"Training accuracy: {train_accuracy_dt:.3f}")
print(f"Test accuracy:     {test_accuracy_dt:.3f}")
print(f"Accuracy gap:      {train_accuracy_dt - test_accuracy_dt:.3f}")

**Write down – Answer:** The default decision tree reaches **1.000 training accuracy** and about **0.776 test accuracy**, a gap of about **0.224**. This large gap strongly suggests overfitting: the unrestricted tree fits the training set almost perfectly but generalizes less well to unseen customers.

## Step 8: Evaluate the Model Properly

We examine the test-set confusion matrix, precision, recall, F1-score, and ROC-AUC. The positive class (`1`) represents a loan default.

In [ ]:
y_test_proba_dt = dt.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_test_pred_dt, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Default (0)", "Default (1)"]).plot(cmap="Blues")
plt.title("Confusion Matrix – Untuned Decision Tree")
plt.tight_layout()
plt.show()

print(classification_report(
    y_test, y_test_pred_dt, labels=[0, 1],
    target_names=["No Default", "Default"], zero_division=0
))
print(f"ROC-AUC: {roc_auc_score(y_test, y_test_proba_dt):.3f}")

**Write down – Answer:** Approving a loan that later defaults is generally costly, but rejecting a reliable customer also has a cost. If the bank's priority is to miss as few defaulters as possible, **recall for the `Default` class** is especially important. Precision must also be considered because low precision means more reliable customers may be incorrectly flagged. The final decision threshold should reflect the bank's actual costs.

## Step 9: Control Overfitting by Tuning

First, we compare tree depths 3, 5, 7, and 10. Then we use 5-fold `GridSearchCV` over `max_depth`, `min_samples_leaf`, and `criterion`. Because the target is imbalanced and identifying defaulters matters, we use F1-score for model selection and `class_weight="balanced"` to give the minority class more influence. The final test set is kept separate from cross-validation.

In [ ]:
depth_results = []
for depth in [3, 5, 7, 10]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth,
        "training_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, model.predict(X_test))
    })

depth_results_df = pd.DataFrame(depth_results)
display(depth_results_df)

param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)
grid.fit(X_train, y_train)

best = grid.best_estimator_
print("Best parameters:", grid.best_params_)
print(f"Best mean CV F1-score: {grid.best_score_:.3f}")

best_pred = best.predict(X_test)
best_proba = best.predict_proba(X_test)[:, 1]
print(f"Tuned tree test accuracy: {accuracy_score(y_test, best_pred):.3f}")
print(f"Tuned tree test precision: {precision_score(y_test, best_pred, zero_division=0):.3f}")
print(f"Tuned tree test recall: {recall_score(y_test, best_pred, zero_division=0):.3f}")
print(f"Tuned tree test F1-score: {f1_score(y_test, best_pred, zero_division=0):.3f}")
print(f"Tuned tree test ROC-AUC: {roc_auc_score(y_test, best_proba):.3f}")

**Write down – Answer:** Among the tested depths 3, 5, 7, and 10, **max_depth = 5** has the highest test accuracy in this run (about **0.826**). Grid search selects **max_depth = 3, min_samples_leaf = 5, criterion = gini** using 5-fold cross-validation F1-score. The tuned tree's test accuracy is lower than the untuned tree's, but its default-class recall is higher (**about 0.514 vs 0.351**). This illustrates why tuning should be judged using the bank's chosen metric, not accuracy alone.

## Step 10: Compare with a Baseline Model

We train Logistic Regression on the same split after standardizing the encoded features, then compare accuracy, recall, F1-score, and ROC-AUC for Logistic Regression, the untuned tree, and the tuned tree.

In [ ]:
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
log_reg.fit(X_train, y_train)

models = {
    "Logistic Regression": log_reg,
    "Untuned Decision Tree": dt,
    "Tuned Decision Tree": best
}

comparison = []
for name, model in models.items():
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Recall (Default)": recall_score(y_test, pred, zero_division=0),
        "F1 (Default)": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })

comparison_df = pd.DataFrame(comparison).set_index("Model").round(3)
display(comparison_df)

**Write down – Answer:** On this reproducible split, Logistic Regression has the strongest overall default-class F1-score (**about 0.472**) and ROC-AUC (**about 0.771**), and recall is about **0.676**. The tuned tree's recall is about **0.514**, while the untuned tree's recall is about **0.351**. Based on these results, Logistic Regression is the better predictive candidate for identifying defaulters in this particular split, while the tuned tree can be useful when simple decision rules are needed. This is a small educational dataset, so validate on more representative data before making lending decisions.

## Step 11: Interpret the Model

We plot the ten most important features of the tuned decision tree and visualize its first three levels for readability.

In [ ]:
importances = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 feature importances:")
display(importances.head(10).to_frame("importance"))

importances.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 Feature Importances – Tuned Decision Tree")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 10))
plot_tree(
    best, feature_names=X.columns, class_names=["No Default", "Default"],
    max_depth=3, filled=True, rounded=True, fontsize=8
)
plt.title("Tuned Decision Tree (First 3 Levels)")
plt.tight_layout()
plt.show()

print("Top five features:", importances.head(5).index.tolist())

**Write down – Answer:** In this run, the tuned tree's top features are **credit_score, loan_amount, employment_years, age**, followed by any remaining feature with non-zero importance (see the printed list). The first three account for most of the tree's importance. Two example rules visible in the tree are: (1) when `credit_score <= 538.5` and `age <= 59`, the tree predicts default; (2) when `credit_score <= 680.5`, `credit_score > 538.5`, and `loan_amount > 235220`, the tree predicts default. These are model rules learned from this dataset, not general financial rules or causal claims.

## Step 12: Conclusion

The following conclusion summarizes the workflow. Update the bracketed parts after reviewing the actual model comparison and feature-importance outputs.

**Conclusion for a Bank Manager (4–6 sentences):**

This project developed a supervised machine-learning model to predict whether a loan applicant may default using historical customer and loan information. The data was inspected, cleaned, encoded, and divided into training and test sets before fitting a Decision Tree and comparing it with Logistic Regression. Model performance was assessed using accuracy, precision, recall, F1-score, ROC-AUC, and a confusion matrix because the target classes are imbalanced. The preferred model should be selected based on the bank's cost of missed defaulters versus incorrectly flagging reliable customers; the comparison table provides evidence for this decision. The model is limited by the size and quality of the available dataset and should not be used as the sole basis for real lending decisions without further validation, fairness checks, and monitoring. A useful next improvement would be to evaluate ensemble models such as Random Forest and validate them on newer, representative loan data.

## Before You Submit

- Run all cells from top to bottom using **Runtime → Run all** (Google Colab) or **Kernel → Restart and Run All** (Jupyter).
- Check that every “Write down” question has a response and update any answer that asks you to report a result from your own outputs.
- Keep `loan_default.csv` in the same working directory as the notebook or upload it to the Colab session.
- Save the notebook as `RollNo_Name_LoanDefault.ipynb`.
- Remember: this is an educational model, not a validated lending decision system.